# Edu-ca-te Data Cleaning
**Beacon Innovation Hub — Data Department Foundational Competence Challenge**

This notebook loads the raw `educate_linked_challenge_dirty_assessment.xlsx` workbook,
applies documented cleaning rules to each sheet, validates the result (row counts,
subject consistency, join integrity), and exports cleaned CSVs plus a cleaning log.

**Before running in Colab:** upload the raw `.xlsx` file using the cell below (or mount
Google Drive and point `RAW_PATH` at it).


In [ ]:
# If running in Google Colab, upload the raw workbook here.
# Comment this cell out if running locally with the file already present.
try:
    from google.colab import files
    uploaded = files.upload()  # select educate_linked_challenge_dirty_assessment.xlsx when prompted
except ImportError:
    print("Not running in Colab — skipping upload widget. Make sure the raw file is in the working directory.")


In [ ]:
import pandas as pd
import numpy as np
import re
import os

RAW_PATH = "educate_linked_challenge_dirty_assessment.xlsx"
OUT_DIR = "cleaned"
os.makedirs(OUT_DIR, exist_ok=True)

# Cleaning log: every action taken is appended here as a row.
cleaning_log = []

def log(dataset, problem, action, justification, count):
    cleaning_log.append({
        "dataset": dataset,
        "problem": problem,
        "action": action,
        "justification": justification,
        "rows_affected": count,
    })


## 1. Load raw data

In [ ]:
xls = pd.ExcelFile(RAW_PATH)
learners = pd.read_excel(xls, "learners")
engagement = pd.read_excel(xls, "engagement")
assessments = pd.read_excel(xls, "assessments")
capacity = pd.read_excel(xls, "support_capacity")
enquiries = pd.read_excel(xls, "enquiries")

raw_counts = {
    "learners": len(learners),
    "engagement": len(engagement),
    "assessments": len(assessments),
    "support_capacity": len(capacity),
    "enquiries": len(enquiries),
}
raw_counts


In [ ]:
excluded_records = []  # rows removed from analysis go here, with a reason — never deleted outright

def flag_excluded(df_name, rows, reason):
    if len(rows) == 0:
        return
    r = rows.copy()
    r["source_dataset"] = df_name
    r["exclusion_reason"] = reason
    excluded_records.append(r)


## 2. Shared cleaning helpers

- **Subject name standardization** — the same subject is spelled many different ways across sheets
  (`Maths`/`Math`/`Mathematics`, `Sci`/`Physical Sciences`/`phys sci`, etc.). All sheets are mapped to
  4 canonical subjects: **Mathematics, Physical Sciences, Life Sciences, Accounting**.
  *Assumption: "Sci" is treated as shorthand for Physical Sciences* — there's no separate "General
  Science" subject in `support_capacity`, and "phys sci" is used elsewhere as an unambiguous
  abbreviation for the same subject.
- **learner_id standardization** — strip whitespace, uppercase.
- **Multi-format date parsing** — the raw dates mix ISO (`YYYY-MM-DD`), `DD/MM/YYYY`, `DD/MM/YY`,
  `DD-MM-YYYY`, and `Month DD YYYY`. Values that still can't be parsed (e.g. `'not_a_date'`, or a
  calendar-invalid date like 30 February) are set to missing rather than guessed.


In [ ]:
SUBJECT_MAP = {
    "maths": "Mathematics", "math": "Mathematics", "mathematics": "Mathematics",
    "physical sciences": "Physical Sciences", "physical science": "Physical Sciences",
    "phys sci": "Physical Sciences",
    "sci": "Physical Sciences",  # ASSUMPTION — see note above
    "life sciences": "Life Sciences", "life science": "Life Sciences", "ls": "Life Sciences",
    "accounting": "Accounting", "acc": "Accounting",
}

def std_subject(x):
    if pd.isna(x):
        return np.nan
    key = str(x).strip().lower()
    return SUBJECT_MAP.get(key, str(x).strip())

def std_learner_id(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip().upper()

def parse_date_multi(x):
    """Try multiple known formats; return NaT if unparseable or calendar-invalid."""
    if pd.isna(x):
        return pd.NaT
    s = str(x).strip()
    formats = ["%Y-%m-%d", "%d/%m/%Y", "%d/%m/%y", "%d-%m-%Y", "%B %d %Y"]
    for fmt in formats:
        try:
            return pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT


## 3. Clean `learners`

Rules applied, in order:
1. Normalize `learner_id` (strip + uppercase)
2. Drop fully duplicate rows
3. Exclude rows with missing `learner_id` (can't be linked) — kept in the flagged file, not deleted
4. Exclude duplicate `learner_id`s after normalization (keep first occurrence)
5. Standardize `gender` to {Female, Male, Non-binary, Not disclosed}
6. Standardize `grade` to {10, 11, 12}; flag any out-of-scope grade rather than dropping it
7. Parse `registration_date` across known formats; unparseable values become missing


In [ ]:
n0 = len(learners)
learners["learner_id"] = learners["learner_id"].apply(std_learner_id)

dup_full = learners.duplicated().sum()
learners = learners.drop_duplicates()
log("learners", "Fully duplicate rows", "Dropped (kept first)",
    "Identical across all columns; a true duplicate record adds no information and would double-count the learner",
    dup_full)

missing_id_mask = learners["learner_id"].isna()
flag_excluded("learners", learners[missing_id_mask], "Missing learner_id - cannot be linked to engagement/assessments")
log("learners", "Missing learner_id", "Excluded from analysis (kept in flagged file)",
    "No key means the row cannot be joined to any other dataset and contributes nothing usable",
    missing_id_mask.sum())
learners = learners[~missing_id_mask]

dup_id_mask = learners.duplicated(subset=["learner_id"], keep="first")
flag_excluded("learners", learners[dup_id_mask], "Duplicate learner_id after normalizing case/whitespace (kept first occurrence)")
log("learners", "Duplicate learner_id (post-normalization)", "Dropped later occurrence, kept first",
    "Same learner_id after standardizing case/whitespace almost certainly represents one learner entered twice; keeping the first avoids double-counting in engagement/assessment joins",
    dup_id_mask.sum())
learners = learners[~dup_id_mask]


In [ ]:
GENDER_MAP = {
    "female": "Female", "f": "Female",
    "male": "Male", "m": "Male",
    "non-binary": "Non-binary",
    "unknown": "Not disclosed", "prefer not say": "Not disclosed",
}
def std_gender(x):
    if pd.isna(x):
        return "Not disclosed"
    key = str(x).strip().lower()
    return GENDER_MAP.get(key, str(x).strip())

learners["gender"] = learners["gender"].apply(std_gender)
log("learners", "Inconsistent gender labels", "Standardized to {Female, Male, Non-binary, Not disclosed}",
    "Free-text and inconsistent-case labels represented the same categories; standardizing enables consistent demographic reporting",
    n0)


In [ ]:
def std_grade(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    s = s.replace("grade", "").replace("gr", "").replace("g", "").strip()
    word_map = {"ten": "10", "eleven": "11", "twelve": "12"}
    for w, num in word_map.items():
        if w in s:
            return int(num)
    digits = re.sub(r"[^0-9]", "", s)
    return int(digits) if digits else np.nan

learners["grade"] = learners["grade"].apply(std_grade)

out_of_scope_mask = ~learners["grade"].isin([10, 11, 12])
learners["in_scope_grade"] = ~out_of_scope_mask
log("learners", "Grade outside Grade 10-12 scope", "Kept record, flagged with in_scope_grade=False",
    "Platform is scoped to Grade 10-12 learners; an out-of-scope grade is retained (not fabricated away) but excluded from grade-specific breakdowns",
    out_of_scope_mask.sum())


In [ ]:
was_present = learners["registration_date"].notna()
learners["registration_date"] = learners["registration_date"].apply(parse_date_multi)
now_missing = learners["registration_date"].isna()
n_unparseable = (was_present & now_missing).sum()
log("learners", "registration_date in mixed/invalid formats", "Parsed using known formats (ISO, DD/MM/YYYY, DD/MM/YY, DD-MM-YYYY, 'Month DD YYYY'); unparseable or calendar-invalid values set to missing (not guessed)",
    "Multiple genuine date formats were present in the source; values like 'not_a_date' or a calendar-invalid date (e.g. 30 Feb) cannot be corrected without fabricating information",
    n_unparseable)

learners_clean = learners.reset_index(drop=True)
learners_clean.head()


## 4. Clean `engagement`

Rules applied:
1. Normalize `learner_id` and `subject`
2. Drop fully duplicate rows
3. Exclude rows with missing `subject`
4. Exclude duplicate learner+subject pairs (same learner's participation recorded twice) — keep first
5. Set `tutoring_attendance_pct` outside 0–100 to missing
6. Set negative `watch_time_minutes` to missing
7. Exclude rows whose `learner_id` has no match in cleaned `learners` (orphaned records) — kept in the flagged file


In [ ]:
n0 = len(engagement)
engagement["learner_id"] = engagement["learner_id"].apply(std_learner_id)
engagement["subject"] = engagement["subject"].apply(std_subject)

dup_full = engagement.duplicated().sum()
engagement = engagement.drop_duplicates()
log("engagement", "Fully duplicate rows", "Dropped (kept first)",
    "Identical rows would double-count a learner's attendance/watch-time for a subject",
    dup_full)

missing_subj_mask = engagement["subject"].isna()
flag_excluded("engagement", engagement[missing_subj_mask], "Missing subject - cannot attribute engagement to a subject")
log("engagement", "Missing subject", "Excluded from analysis (kept in flagged file)",
    "Subject-level engagement cannot be computed or attributed without a subject value",
    missing_subj_mask.sum())
engagement = engagement[~missing_subj_mask]

dup_pair_mask = engagement.duplicated(subset=["learner_id", "subject"], keep="first")
flag_excluded("engagement", engagement[dup_pair_mask], "Duplicate learner+subject engagement record (kept first occurrence)")
log("engagement", "Duplicate learner+subject pairs", "Dropped later occurrence, kept first",
    "Same learner's engagement for the same subject recorded twice would double-count participation",
    dup_pair_mask.sum())
engagement = engagement[~dup_pair_mask]


In [ ]:
att = pd.to_numeric(engagement["tutoring_attendance_pct"], errors="coerce")
bad_att_mask = (att < 0) | (att > 100)
engagement.loc[bad_att_mask, "tutoring_attendance_pct"] = np.nan
log("engagement", "tutoring_attendance_pct outside 0-100", "Set to missing (excluded from attendance metrics)",
    "A percentage outside 0-100 is not physically valid and the true value cannot be inferred; treating as missing avoids fabricating a number",
    bad_att_mask.sum())

wt = pd.to_numeric(engagement["watch_time_minutes"], errors="coerce")
bad_wt_mask = wt < 0
engagement.loc[bad_wt_mask, "watch_time_minutes"] = np.nan
log("engagement", "Negative watch_time_minutes", "Set to missing (excluded from watch-time metrics)",
    "Negative minutes are not physically possible and the correct value cannot be inferred",
    bad_wt_mask.sum())

engagement["tutoring_attendance_pct"] = pd.to_numeric(engagement["tutoring_attendance_pct"], errors="coerce")
engagement["watch_time_minutes"] = pd.to_numeric(engagement["watch_time_minutes"], errors="coerce")
engagement["videos_watched"] = pd.to_numeric(engagement["videos_watched"], errors="coerce")


In [ ]:
valid_ids = set(learners_clean["learner_id"])
orphan_mask = ~engagement["learner_id"].isin(valid_ids)
flag_excluded("engagement", engagement[orphan_mask], "learner_id not found in learners master table")
log("engagement", "Orphaned learner_id (no matching learner record)", "Excluded from analysis (kept in flagged file)",
    "Cannot attribute engagement to a known learner; per project decision, orphaned rows are excluded from analysis but retained in a flagged file rather than deleted",
    orphan_mask.sum())
engagement = engagement[~orphan_mask]

engagement_clean = engagement.reset_index(drop=True)
engagement_clean.head()


## 5. Clean `assessments`

Rules applied:
1. Normalize `learner_id` and `subject`
2. Drop fully duplicate rows
3. Exclude rows missing `learner_id`/`subject`
4. Coerce non-numeric `baseline_mark` (e.g. `'MISSING'`) to missing
5. Set out-of-range `baseline_mark`/`latest_mark` (outside 0–100) to missing — **not** guess-corrected.
   The repeated values of exactly 1050 (×9) and 180 (×2) in `latest_mark` look like a systematic
   entry/extraction error, but the true value can't be reconstructed with confidence, so they're
   treated as missing rather than "fixed" by assumption.
6. Exclude orphaned `learner_id`s
7. Derive `mark_change = latest_mark - baseline_mark` (only valid where both marks are present)


In [ ]:
n0 = len(assessments)
assessments["learner_id"] = assessments["learner_id"].apply(std_learner_id)
assessments["subject"] = assessments["subject"].apply(std_subject)

dup_full = assessments.duplicated().sum()
assessments = assessments.drop_duplicates()
log("assessments", "Fully duplicate rows", "Dropped (kept first)",
    "Identical rows would double-count a learner's marks for a subject",
    dup_full)

missing_core_mask = assessments[["learner_id", "subject"]].isna().any(axis=1)
flag_excluded("assessments", assessments[missing_core_mask], "Missing learner_id and/or subject - row cannot be attributed")
log("assessments", "Missing learner_id/subject", "Excluded from analysis (kept in flagged file)",
    "Without a learner and subject key the row cannot be joined or attributed to any KPI",
    missing_core_mask.sum())
assessments = assessments[~missing_core_mask]


In [ ]:
bm = pd.to_numeric(assessments["baseline_mark"], errors="coerce")
n_bm_nonnumeric = (assessments["baseline_mark"].notna() & bm.isna()).sum()
bad_bm_range = (bm < 0) | (bm > 100)
bm[bad_bm_range] = np.nan
assessments["baseline_mark"] = bm
log("assessments", "baseline_mark non-numeric text (e.g. 'MISSING')", "Coerced to missing",
    "A text placeholder in a numeric mark field cannot be treated as a value; recorded as missing rather than guessed",
    n_bm_nonnumeric)
log("assessments", "baseline_mark outside 0-100", "Set to missing (excluded from mark-based metrics)",
    "A mark outside the valid 0-100 range is not usable and the correct value cannot be inferred",
    bad_bm_range.sum())

lm = pd.to_numeric(assessments["latest_mark"], errors="coerce")
bad_lm_range = (lm < 0) | (lm > 100)
lm[bad_lm_range] = np.nan
assessments["latest_mark"] = lm
log("assessments", "latest_mark outside 0-100 (repeated values of 1050 and 180)", "Set to missing (excluded from mark-based metrics)",
    "Values of exactly 1050 (x9) and 180 (x2) are implausible marks and look like a systematic entry/extraction error (e.g. stray digit or decimal shift); the true value cannot be reconstructed with confidence, so it is treated as missing rather than 'corrected' by guesswork",
    bad_lm_range.sum())


In [ ]:
orphan_mask = ~assessments["learner_id"].isin(valid_ids)
flag_excluded("assessments", assessments[orphan_mask], "learner_id not found in learners master table")
log("assessments", "Orphaned learner_id (no matching learner record)", "Excluded from analysis (kept in flagged file)",
    "Cannot attribute marks to a known learner; excluded from analysis but retained in flagged file",
    orphan_mask.sum())
assessments = assessments[~orphan_mask]

assessments["mark_change"] = assessments["latest_mark"] - assessments["baseline_mark"]

assessments_clean = assessments.reset_index(drop=True)
assessments_clean.head()


## 6. Clean `support_capacity`

The raw sheet has 8 rows for what should be 4 subjects — each subject appears twice, once cleanly
and once with a messy label or a messy value in one field (e.g. `"26 hrs"` instead of `26`, or a
negative `-2`). The fix: clean numeric fields first, then merge duplicate subject rows, taking the
first valid (non-null, in-range) value per column across the pair.


In [ ]:
capacity["subject"] = capacity["subject"].apply(std_subject)

def clean_hours(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower().replace("hrs", "").replace("hr", "").strip()
    return pd.to_numeric(s, errors="coerce")

def clean_cost(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper().replace("R", "").strip()
    return pd.to_numeric(s, errors="coerce")

capacity["weekly_tutor_hours"] = capacity["weekly_tutor_hours"].apply(clean_hours)
capacity["estimated_cost_per_extra_hour"] = capacity["estimated_cost_per_extra_hour"].apply(clean_cost)

neg_hours_mask = capacity["weekly_tutor_hours"] < 0
capacity.loc[neg_hours_mask, "weekly_tutor_hours"] = np.nan
log("support_capacity", "Negative weekly_tutor_hours", "Set to missing prior to merge (recovered from duplicate subject row where available)",
    "Negative tutoring hours are not physically meaningful",
    neg_hours_mask.sum())

n_rows_before_merge = len(capacity)
capacity_clean = (
    capacity.groupby("subject", as_index=False)
    .agg({
        "active_tutors": "first",
        "weekly_tutor_hours": lambda s: s.dropna().iloc[0] if s.dropna().shape[0] else np.nan,
        "waiting_list": lambda s: s.dropna().iloc[0] if s.dropna().shape[0] else np.nan,
        "estimated_cost_per_extra_hour": lambda s: s.dropna().iloc[0] if s.dropna().shape[0] else np.nan,
    })
)
log("support_capacity", "Duplicate/inconsistent subject rows (8 rows for 4 real subjects)",
    f"Merged into {len(capacity_clean)} canonical subject records, taking the first valid (non-null, in-range) value per field across duplicates",
    "Each subject had two near-duplicate rows differing only in label formatting or a single messy field; merging preserves the underlying capacity data without double-counting tutors/hours in any join",
    n_rows_before_merge - len(capacity_clean))

capacity_clean = capacity_clean.reset_index(drop=True)
capacity_clean


## 7. Clean `enquiries`

Rules applied:
1. Normalize `subject`
2. Standardize `grade`; flag out-of-scope grades (e.g. 13)
3. Standardize `converted_to_registration` to {Yes, No, Maybe, Unknown} — `Maybe` and blanks are
   kept distinct rather than forced into Yes/No, since they're genuinely ambiguous
4. Drop fully duplicate rows / duplicate `enquiry_id`
5. Parse `enquiry_date` across known formats


In [ ]:
n0 = len(enquiries)
enquiries["subject"] = enquiries["subject"].apply(std_subject)

def std_enq_grade(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower().replace("grade", "").strip()
    digits = re.sub(r"[^0-9]", "", s)
    return int(digits) if digits else np.nan

enquiries["grade"] = enquiries["grade"].apply(std_enq_grade)
out_of_scope_mask = ~enquiries["grade"].isin([10, 11, 12])
enquiries["in_scope_grade"] = ~out_of_scope_mask
log("enquiries", "Grade outside Grade 10-12 scope", "Kept record, flagged with in_scope_grade=False",
    "Platform is scoped to Grade 10-12; out-of-scope grade retained but excluded from grade-specific views",
    out_of_scope_mask.sum())


In [ ]:
def std_converted(x):
    if pd.isna(x):
        return "Unknown"
    s = str(x).strip().lower()
    if s in ("y", "yes", "1"):
        return "Yes"
    if s in ("n", "no", "0"):
        return "No"
    if s == "maybe":
        return "Maybe"
    return "Unknown"

enquiries["converted_to_registration"] = enquiries["converted_to_registration"].apply(std_converted)
log("enquiries", "Inconsistent converted_to_registration encoding (Y/Yes/1, N/No/0, Maybe, blank)",
    "Standardized to {Yes, No, Maybe, Unknown}",
    "Multiple encodings represented the same underlying flag; 'Maybe' is kept distinct rather than forced into Yes/No since it is genuinely ambiguous and excluding it (not guessing) preserves conversion-rate accuracy",
    n0)


In [ ]:
dup_full = enquiries.duplicated().sum()
dup_id = enquiries.duplicated(subset=["enquiry_id"], keep="first").sum()
enquiries = enquiries.drop_duplicates()
enquiries = enquiries[~enquiries.duplicated(subset=["enquiry_id"], keep="first")]
log("enquiries", "Fully duplicate rows / duplicate enquiry_id", "Dropped later occurrence, kept first",
    "Same enquiry recorded twice would double-count demand or conversion for that subject",
    max(dup_full, dup_id))

was_present = enquiries["enquiry_date"].notna()
enquiries["enquiry_date"] = enquiries["enquiry_date"].apply(parse_date_multi)
now_missing = enquiries["enquiry_date"].isna()
n_unparseable = (was_present & now_missing).sum()
log("enquiries", "enquiry_date in mixed/invalid formats (incl. calendar-invalid dates like 31 June)", "Parsed using known formats; unparseable/invalid set to missing",
    "Mixed date formats and at least one calendar-impossible date were present; cannot be corrected without fabricating a date",
    n_unparseable)

enquiries_clean = enquiries.reset_index(drop=True)
enquiries_clean.head()


## 8. Export cleaned data, flagged/excluded records, and the cleaning log

In [ ]:
learners_clean.to_csv(f"{OUT_DIR}/learners_clean.csv", index=False)
engagement_clean.to_csv(f"{OUT_DIR}/engagement_clean.csv", index=False)
assessments_clean.to_csv(f"{OUT_DIR}/assessments_clean.csv", index=False)
capacity_clean.to_csv(f"{OUT_DIR}/support_capacity_clean.csv", index=False)
enquiries_clean.to_csv(f"{OUT_DIR}/enquiries_clean.csv", index=False)

if excluded_records:
    excluded_df = pd.concat(excluded_records, ignore_index=True, sort=False)
else:
    excluded_df = pd.DataFrame()
excluded_df.to_csv(f"{OUT_DIR}/excluded_flagged_records.csv", index=False)

log_df = pd.DataFrame(cleaning_log)
log_df.to_csv(f"{OUT_DIR}/cleaning_log.csv", index=False)

print(f"Files written to ./{OUT_DIR}/")
log_df


## 9. Validation

- Row counts before/after
- Subject consistency check (should be exactly 4 canonical subjects everywhere)
- Join integrity check (confirms no row multiplication or silent loss when joining `learners` to
  `engagement`/`assessments`)


In [ ]:
print("ROW COUNTS: raw -> cleaned")
print(f"learners:          {raw_counts['learners']:>4} -> {len(learners_clean)}")
print(f"engagement:        {raw_counts['engagement']:>4} -> {len(engagement_clean)}")
print(f"assessments:       {raw_counts['assessments']:>4} -> {len(assessments_clean)}")
print(f"support_capacity:  {raw_counts['support_capacity']:>4} -> {len(capacity_clean)}")
print(f"enquiries:         {raw_counts['enquiries']:>4} -> {len(enquiries_clean)}")
print(f"excluded/flagged rows retained: {len(excluded_df)}")


In [ ]:
print("SUBJECT CONSISTENCY CHECK (should be exactly 4 canonical subjects each)")
for name, df in [("engagement", engagement_clean), ("assessments", assessments_clean),
                  ("support_capacity", capacity_clean), ("enquiries", enquiries_clean)]:
    subs = sorted(df["subject"].dropna().unique())
    print(f"{name}: {len(subs)} subjects -> {subs}")


In [ ]:
print("JOIN INTEGRITY CHECK: learners <-> engagement/assessments")
merged_eng = engagement_clean.merge(learners_clean, on="learner_id", how="left", indicator=True)
print("engagement rows before join:", len(engagement_clean), "| after left-join to learners:", len(merged_eng),
      "| unmatched:", (merged_eng["_merge"] == "left_only").sum())

merged_ass = assessments_clean.merge(learners_clean, on="learner_id", how="left", indicator=True)
print("assessments rows before join:", len(assessments_clean), "| after left-join to learners:", len(merged_ass),
      "| unmatched:", (merged_ass["_merge"] == "left_only").sum())


## 10. (Colab) Download the cleaned files

Run this cell to download the cleaned CSVs, the flagged/excluded records file, and the cleaning
log to your machine — ready to add to your GitHub repo.


In [ ]:
try:
    from google.colab import files
    import shutil
    zip_path = shutil.make_archive("cleaned_educate_data", "zip", OUT_DIR)
    files.download(zip_path)
except ImportError:
    print("Not running in Colab — cleaned files are already in the ./cleaned/ folder.")
